In [1]:
!git clone https://github.com/dshipley71/mcp-rag.git

Cloning into 'mcp-rag'...
remote: Enumerating objects: 309, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 309 (delta 69), reused 0 (delta 0), pack-reused 184 (from 1)
Receiving objects: 100% (309/309), 141.22 KiB | 5.23 MiB/s, done.
Resolving deltas: 100% (159/159), done.


In [2]:
%cd mcp-rag

/content/mcp-rag


In [3]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 339.2/339.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 705.5/705.5 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.3/123.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33

In [4]:
!apt-get remove -y nodejs npm
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!node -v
!npm -v

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'npm' is not installed, so not removed
Package 'nodejs' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 7 not upgraded.
2026-04-02 19:22:20 - Installing pre-requisites
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,968 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backport

In [5]:
from pathlib import Path

REPO_ROOT = Path("/content/mcp-rag").resolve()
DOCS_DIR = REPO_ROOT / "docs"
DB_DIR = REPO_ROOT / ".rag" / "velocirag"

print("REPO_ROOT:", REPO_ROOT)
print("DOCS_DIR:", DOCS_DIR)
print("DB_DIR:", DB_DIR)

REPO_ROOT: /content/mcp-rag
DOCS_DIR: /content/mcp-rag/docs
DB_DIR: /content/mcp-rag/.rag/velocirag


In [6]:
import shutil
import subprocess

if DB_DIR.exists():
    shutil.rmtree(DB_DIR)

result = subprocess.run(
    ["velocirag", "index", str(DOCS_DIR)],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)

print("RETURN CODE:", result.returncode)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

RETURN CODE: 0
STDOUT:
 Found 1 markdown files
Indexing /content/mcp-rag/docs...

Results:
  Files processed: 1
  Files skipped: 0 (unchanged)
  Chunks added: 19
  Total documents: 19
  Processing time: 8.8s

Building knowledge graph...
Graph build complete:
  Nodes: 2
  Edges: 1
  Time: 0.5s

Metadata extraction:
  Documents: 1
  Tags extracted: 0
  Cross-refs: 0

Index ready for search.

STDERR:



In [7]:
import nest_asyncio
nest_asyncio.apply()

In [8]:
from src.mcp_runtime import MCPRuntime

runtime = MCPRuntime(
    db_dir=str(DB_DIR),
    docs_dir=str(DOCS_DIR),
)

await runtime.connect()

print("VelociRAG tools:", await runtime.velocirag.list_tools())
print("Filesystem tools:", await runtime.filesystem.list_tools())

VelociRAG tools: ['search', 'index', 'add_document', 'health', 'list_sources']
Filesystem tools: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories']


In [9]:
import rich

health = await runtime.velocirag.call_tool("health", {})
rich.print("HEALTH:", health)

HEALTH:
CallToolResult(
    meta=None,
    content=[
        TextContent(
            type='text',
            text='{"total_documents":0,"total_chunks":0,"index_dimensions":null,"graph_nodes":0,"graph_edges":0,"la
yers":[],"model_name":"all-MiniLM-L6-v2","db_path":"/content/mcp-rag/.rag/velocirag","components":{"vector_store":t
rue,"graph_store":false,"metadata_store":false,"unified_search":true}}',
            annotations=None,
            meta=None
        )
    ],
    structuredContent={
        'total_documents': 0,
        'total_chunks': 0,
        'index_dimensions': None,
        'graph_nodes': 0,
        'graph_edges': 0,
        'layers': [],
        'model_name': 'all-MiniLM-L6-v2',
        'db_path': '/content/mcp-rag/.rag/velocirag',
        'components': {
            'vector_store': True,
            'graph_store': False,
            'metadata_store': False,
            'unified_search': True
        }
    },
    isError=False
)

In [10]:
search = await runtime.velocirag.call_tool(
    "search",
    {"query": "VelociRAG", "limit": 3}
)
rich.print("SEARCH:", search)

SEARCH:
CallToolResult(
    meta=None,
    content=[
        TextContent(
            type='text',
            text='{"error":"No documents indexed. Use the index tool to add documents 
first.","results":[],"total_results":0,"search_time_ms":0}',
            annotations=None,
            meta=None
        )
    ],
    structuredContent={
        'error': 'No documents indexed. Use the index tool to add documents first.',
        'results': [],
        'total_results': 0,
        'search_time_ms': 0
    },
    isError=False
)

In [11]:
result = await runtime.filesystem.call_tool(
    "read_text_file",
    {"path": str(DOCS_DIR / "VelociRAG.md")}
)
rich.print("FILESYSTEM READ:", result)

FILESYSTEM READ:
CallToolResult(
    meta=None,
    content=[
        TextContent(
            type='text',
            text='# 🦖 VelociRAG\n\n**Lightning-fast RAG for AI agents.**\n\n_Four-layer retrieval fusion powered 
by ONNX Runtime. No PyTorch. Sub-200ms warm search. Incremental graph updates. MCP-ready._\n\n---\n\nMost RAG 
solutions either drag in 2GB+ of PyTorch or limit you to single-layer vector search. VelociRAG gives you four 
retrieval methods — vector similarity, BM25 keyword matching, knowledge graph traversal, and metadata filtering — 
fused through reciprocal rank fusion with cross-encoder reranking. All running on ONNX Runtime, no GPU, no API 
keys. Comes with an MCP server for agent integration, a Unix socket daemon for warm queries, and a CLI that just 
works.\n\n## 🚀 Quick Start\n\n### MCP Server (Claude, Cursor, Windsurf)\n\n```bash\npip install 
"velocirag[mcp]"\nvelocirag index ./my-docs\nvelocirag mcp\n```\n\n**Claude Code** — add to `.mcp.json` in your 
project root:\n```json\n{\n  "mcpServers": {\n    "velocirag": {\n      "command": "velocirag",\n      "args": 
["mcp"],\n      "env": { "VELOCIRAG_DB": "/path/to/data" }\n    }\n  }\n}\n```\nThen open `/mcp` in Claude Code and
enable the `velocirag` server. If using a virtualenv, use the full path to the binary (e.g. 
`.venv/bin/velocirag`).\n\n**Claude Desktop** — add to `claude_desktop_config.json`:\n```json\n{\n  "mcpServers": 
{\n    "velocirag": {\n      "command": "velocirag",\n      "args": ["mcp", "--db", "/path/to/data"]\n    }\n  
}\n}\n```\n\n**Cursor** — add to `.cursor/mcp.json`:\n```json\n{\n  "mcpServers": {\n    "velocirag": {\n      
"command": "velocirag",\n      "args": ["mcp", "--db", "/path/to/data"]\n    }\n  }\n}\n```\n\n### Python 
API\n\n```python\nfrom velocirag import Embedder, VectorStore, Searcher\n\nembedder = Embedder()\nstore = 
VectorStore(\'./my-db\', embedder)\nstore.add_directory(\'./my-docs\')\nsearcher = Searcher(store, 
embedder)\nresults = searcher.search(\'query\', limit=5)\n```\n\n### CLI\n\n```bash\npip install 
velocirag\nvelocirag index ./my-docs\nvelocirag search "your query here"\n```\n\n### Search Daemon (warm engine for
CLI users)\n\n```bash\nvelocirag serve --db ./my-data        # start daemon (background)\nvelocirag search "query" 
# auto-routes through daemon\nvelocirag status                      # check daemon health\nvelocirag stop          
# stop daemon\n```\n\nThe daemon keeps the ONNX model + FAISS index warm over a Unix socket. First query loads the 
engine (~1s), subsequent queries return in ~180ms with full 4-layer fusion.\n\n## 🎯 Why VelociRAG?\n\n| | 
VelociRAG | LangChain | LlamaIndex | Chroma | mcp-local-rag |\n|---|:---:|:---:|:---:|:---:|:---:|\n| **Search 
layers** | 4 | 2 | 2 | 1 | 2 |\n| **Cross-encoder reranking** | ✅ | ❌ | ✅ | ❌ | ❌ |\n| **Knowledge graph** | 
✅ | ❌ | ✅ | ❌ | ❌ |\n| **Incremental updates** | ✅ | ❌ | ❌ | ❌ | ❌ |\n| **LLM required for search** | ❌ 
| ⚠️ | ⚠️ | ❌ | ❌ |\n| **MCP server** | ✅ | ❌ | ❌ | ❌ | ✅ |\n| **GPU required** | ❌ | ❌ | ❌ | ❌ | ❌ |\n| 
**PyTorch required** | ❌ | ✅ | ✅ | ❌ | ❌ |\n| **Install size** | ~80MB | ~750MB+ | ~750MB+ | ~50MB | ~30MB 
|\n| **Warm search latency** | ~3ms | — | — | ~50ms | ~200ms |\n\n## 🏗️ How It Works\n\n**The 4-layer 
pipeline:**\n```\nQuery → expand (acronyms, variants)\n      → [Vector]   FAISS cosine similarity (384d, 
MiniLM-L6-v2 via ONNX)\n      → [Keyword]  BM25 via SQLite FTS5\n      → [Graph]    Knowledge graph traversal\n    
→ [Metadata] Structured SQL filters (tags, status, project)\n      → RRF Fusion → Cross-encoder rerank → 
Results\n```\n\n**What each layer catches:**\n\n| Query type | Vector | Keyword | Graph | Metadata 
|\n|-----------|:---:|:---:|:---:|:---:|\n| Conceptual ("improve error handling") | ✅ | — | — | — |\n| Exact match
("ERR_CONNECTION_REFUSED") | — | ✅ | — | — |\n| Connected concepts | — | — | ✅ | — |\n| Filtered ("#python 
status:active") | — | — | — | ✅ |\n| Combined ("React state managem

In [12]:
from src.retrieval import run_bm25_search, run_vector_search, fetch_documents

bm25 = await run_bm25_search(runtime, "What is VelociRAG?")
vector = await run_vector_search(runtime, "What is VelociRAG?")

print("BM25:", bm25[:2])
print("VECTOR:", vector[:2])

combined = bm25 + vector
chunks = await fetch_documents(runtime, combined[:2])

print("NUM CHUNKS:", len(chunks))
if chunks:
    print("FIRST CHUNK PREVIEW:\n")
    print(chunks[0].text[:1000])

BM25: []
VECTOR: []
NUM CHUNKS: 0


In [13]:
from src.orchestrator import run_query
from src.models import QueryRequest

result = await run_query(
    QueryRequest(query="What is VelociRAG?"),
    runtime
)
print(result)

AnswerResult(answer='', citations=[], status='no_evidence')


In [14]:
# Optional: only run at the very end of the notebook session
try:
    await runtime.close()
except BaseException:
    pass